# Capítulo 2 — Representação dos dados: hash, embeddings, cruzamentos e multimodal

## Missão
Capítulo 2, padrões 1 a 4. O id do restaurante tem 1.600 valores e cresce todo mês: o one-hot explode e quebra quando aparece um restaurante novo. Você vai aplicar:
- **Hashed Feature**: um número fixo de baldes, que aceita valores novos;
- **Embeddings**: o restaurante vira um vetor pequeno aprendido;
- **Feature Cross**: hora × dia da semana, para o modelo linear entender o pico;
- **Multimodal Input**: números + texto do comentário no mesmo modelo.

**No fim deste caderno você terá:** o módulo `meu_mlops/representacao.py` (balde por hash estável, cruzamento e embeddings por SVD) e a tabela de MAE de cada representação.

**Leitura essencial (opcional, livro em inglês):**
- Simple Data Representations · PDF p. 43
- Design Pattern 1: Hashed Feature · PDF p. 55
- Design Pattern 2: Embeddings · PDF p. 63
- Design Pattern 3: Feature Cross · PDF p. 79
- Design Pattern 4: Multimodal Input · PDF p. 91

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap02_representacao_dos_dados_hash_embeddings_cruza.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_mlops/
os.makedirs("../meu_mlops", exist_ok=True)
open("../meu_mlops/__init__.py", "a").close()        # meu_mlops/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_mlops")

## Ideia em 30 segundos
- **Hash:** `balde = hash_estável(id) % N`. Valores novos caem em algum balde, e as colisões são o preço.
- **Embedding:** cada categoria vira um vetor denso de poucas dimensões, aprendido para que categorias parecidas fiquem próximas.
- **Cruzamento:** cria a combinação explícita (`sexta_20h`). Um modelo linear não descobre interações sozinho.
- **Multimodal:** concatena representações de tipos diferentes (números, texto, imagem).

### 2.1 **As representações.** Crie `representacao.py` com:
- `balde_hash(valor, n)`: estável entre execuções (`hashlib.md5` em vez de `hash()`, que muda a cada processo);
- `cruzar(df, a, b)`: uma coluna de texto `"a_b"`;
- `embeddings_restaurante(df, k=6)`: monta a matriz restaurante × (hora, dia) com o tempo médio de entrega (centrado), aplica `TruncatedSVD(k)` e devolve um DataFrame `restaurante_id → emb_0..emb_k-1`.

_A célula grava `meu_mlops/representacao.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_mlops/representacao.py


### 2.2 **Hash × one-hot × nada.** Prever o tempo com as numéricas + o restaurante como (a) nada, (b) one-hot e (c) hash com 100 baldes (one-hot do balde), usando `Ridge`. Compare o MAE e o número de colunas. Depois simule **restaurantes novos** no teste (troque o id de 10% dos pedidos por ids nunca vistos): o one-hot ignora, e o hash ainda dá alguma informação?

### 2.3 **Embeddings.** Calcule embeddings de 6 dimensões dos restaurantes **só com o treino** e acrescente-os às numéricas (restaurante sem embedding recebe zeros). Compare o MAE do `Ridge` com o one-hot: o modelo usa 6 colunas em vez de 1.600. Mostre também os 3 restaurantes mais parecidos com o mais popular (distância entre embeddings).

### 2.4 **Feature cross.** O pico (sexta a domingo, 19h a 21h) é uma **interação**. Compare um `Ridge` com hora e dia como números, depois como one-hot separados e, por fim, com o cruzamento `hora × dia_semana` em one-hot. Só o cruzamento deixa o modelo linear capturar o pico.

### 2.5 **Multimodal: números + texto.** Agora o alvo é a **nota** do pedido (1 a 5), prevista com o tempo de entrega e o **texto** do comentário juntos: `ColumnTransformer` com `TfidfVectorizer` no texto e `StandardScaler` nos números, e uma `LogisticRegression`. Compare com só números e só texto (acurácia).

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
